In [5]:
!pip install streamlit scikit-learn pandas
!pip install gspread oauth2client
!pip install transformers


In [3]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pandas as pd



In [4]:
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
creds = ServiceAccountCredentials.from_json_keyfile_name("rare-shadow-446414-p6-8b40475c26af.json", scope)
client = gspread.authorize(creds)

sheet = client.open("Inbox Log").worksheet("الورقة1")

data = sheet.get_all_records()
df = pd.DataFrame(data)

df.head()

,ID,snippet,Date
0,19697ecbdc4c53c1\n,أضف الأشخاص الذين تعرفهم لرؤية صورهم وتحديثاته...,1746305588000
1,196980291cb16567\n,"Hello Hey Nashmico, DMT_Raghad just went live!...",1746307026000
2,1969857340fefcc1\n,"Hello Mohammad Alsharea, I hope this message f...",1746312561000


In [6]:
from transformers import pipeline
classifier = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cuda:0


In [8]:
df["Sentiment"] = df["snippet"].apply(lambda text: classifier(text)[0]["label"])
df.head()

,ID,snippet,Date,Sentiment
0,19697ecbdc4c53c1\n,أضف الأشخاص الذين تعرفهم لرؤية صورهم وتحديثاته...,1746305588000,NEGATIVE
1,196980291cb16567\n,"Hello Hey Nashmico, DMT_Raghad just went live!...",1746307026000,POSITIVE
2,1969857340fefcc1\n,"Hello Mohammad Alsharea, I hope this message f...",1746312561000,POSITIVE


In [9]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0


In [10]:
labels = ["important", "not important"]

In [12]:
def classify_email(text):
    result = classifier(text, labels)
    return result["labels"][0]

df["Category"] = df["snippet"].apply(classify_email)
df.head()

,ID,snippet,Date,Sentiment,Category
0,19697ecbdc4c53c1\n,أضف الأشخاص الذين تعرفهم لرؤية صورهم وتحديثاته...,1746305588000,NEGATIVE,important
1,196980291cb16567\n,"Hello Hey Nashmico, DMT_Raghad just went live!...",1746307026000,POSITIVE,important
2,1969857340fefcc1\n,"Hello Mohammad Alsharea, I hope this message f...",1746312561000,POSITIVE,important
